In [2]:
# @title 0) 저장소 클론·pip 설치 (로컬에서는 생략 가능)
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ▼▼▼ 푸시한 뒤 본인 저장소 URL로 수정 ▼▼▼
REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
# Colab 기본. 바꾸려면: os.environ["COLAB_REPO_DIR"] = "/content/my_colab"
WORKDIR = Path(os.environ.get("COLAB_REPO_DIR", "/content/colab"))

MARK = WORKDIR / "src" / "mindscopex_analysis" / "__init__.py"


def _run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if MARK.is_file():
    print(f"이미 클론됨 → git pull 만 실행: {WORKDIR}")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=False)
else:
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    _run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)])

os.environ["MINDSCOPEX_ROOT"] = str(WORKDIR.resolve())
os.chdir(WORKDIR)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "torch",
        "transformers",
        "accelerate",
        "plotly",
        "pandas",
        "pyyaml",
        "tqdm",
    ]
)
print("pip install -e . 완료")

이미 클론됨 → git pull 만 실행: /content/colab
cwd = /content/colab
MINDSCOPEX_ROOT = /content/colab
pip install -e . 완료


# Reasoning vs Non-Reasoning: 뉴런 활성화 비교

프롬프트 그룹별 레이어·뉴런 활성화를 수집하고 비교합니다.

In [19]:
import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from copy import deepcopy
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype"
print("Plotly renderer:", pio.renderers.default)

# 프로젝트 루트 탐색 (MINDSCOPEX_ROOT 또는 상위 경로)
_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
        raise FileNotFoundError(
            f"MINDSCOPEX_ROOT={env!r} 아래에 {_REPO_MARK.as_posix()} 가 없습니다."
        )
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError(
        "src/mindscopex_analysis 을 찾지 못했습니다. Colab에서는 clone 후 %cd 로 프로젝트 루트로 이동하거나 "
        "os.environ['MINDSCOPEX_ROOT']='/path/to/colab' 을 설정하세요."
    )


_SRC = _find_repo_root() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from mindscopex_analysis.neurons import (
    per_neuron_stats,
    top_k_neurons,
    differential_neurons,
    cosine_similarity_matrix,
)
import mindscopex_analysis.neurons as _neurons_mod
from mindscopex_analysis.sae.trainer import SAETrainer, SAETrainConfig
from mindscopex_analysis.notebook_utils import (
    project_root_from_notebook,
    deep_merge,
    dtype_from_str,
    resolve_target_layer,
)
from mindscopex_analysis.notebook_presets import (
    DEFAULT_EXPERIMENT_BASE,
    MODEL_PRESETS,
    PRESET_CHOICES,
)

assert getattr(_neurons_mod, "COSINE_SIM_IMPL_VERSION", 0) >= 2, (
    "neurons.py 가 구버전입니다. git pull 후 pip install -e . 하고 커널을 재시작하세요."
)
print("준비 완료.")

Plotly renderer: plotly_mimetype
준비 완료.


## 0. 실험 설정

프리셋·YAML·프롬프트 그룹과 분석 레이어를 정합니다.

In [4]:
ROOT = project_root_from_notebook(Path.cwd())
YAML_OVERRIDE = ROOT / "configs" / "notebook_neuron_compare.yaml"

# 프리셋
ACTIVE_PRESET = "qwen_reasoning_pair"

if ACTIVE_PRESET not in MODEL_PRESETS:
    raise ValueError(f"알 수 없는 프리셋 {ACTIVE_PRESET!r}. 가능: {PRESET_CHOICES}")

yaml_data: dict = {}
if YAML_OVERRIDE.is_file():
    with YAML_OVERRIDE.open(encoding="utf-8") as f:
        yaml_data = yaml.safe_load(f) or {}
    if isinstance(yaml_data.get("preset"), str):
        ACTIVE_PRESET = yaml_data["preset"]
        if ACTIVE_PRESET not in MODEL_PRESETS:
            raise ValueError(f"YAML preset {ACTIVE_PRESET!r} 가 MODEL_PRESETS 에 없습니다.")

EXP = deep_merge(deepcopy(DEFAULT_EXPERIMENT_BASE), MODEL_PRESETS[ACTIVE_PRESET])
EXP = deep_merge(EXP, {k: v for k, v in yaml_data.items() if k != "preset"})

MODELS = EXP["models"]
PROMPTS = deepcopy(EXP["prompt_groups"])
_mpg = EXP["analysis"]["max_prompts_per_group"]
if _mpg is not None:
    for _k in list(PROMPTS.keys()):
        PROMPTS[_k] = PROMPTS[_k][:_mpg]

PROMPT_KEYS = list(PROMPTS.keys())
if len(PROMPT_KEYS) < 2:
    raise ValueError("프롬프트 그룹이 2개 이상이어야 비교(P0 vs P1)가 가능합니다.")
P0, P1 = PROMPT_KEYS[0], PROMPT_KEYS[1]

DEVICE = EXP["runtime"]["device"] or ("cuda" if torch.cuda.is_available() else "cpu")
print(f"ACTIVE_PRESET = {ACTIVE_PRESET!r}   DEVICE = {DEVICE}")
print("모델:", MODELS)
print("프롬프트 그룹:", {k: len(v) for k, v in PROMPTS.items()}, "  비교:", repr(P0), "vs", repr(P1))
print("analysis:", EXP["analysis"])

ACTIVE_PRESET = 'qwen_reasoning_pair'   DEVICE = cuda
모델: {'non_reasoning': 'Qwen/Qwen2.5-0.5B-Instruct', 'reasoning': 'Qwen/Qwen3-0.6B'}
프롬프트 그룹: {'intuitive': 1, 'logical': 1}   비교: 'intuitive' vs 'logical'
analysis: {'target_layer': 'mid', 'heatmap_top_neurons': 200, 'differential_top_k': 30, 'sae_top_features': 50, 'max_prompts_per_group': None, 'sae_train': {'epochs': 3, 'k': 32, 'lr': 0.0003, 'l1_coeff': 0.0}}


## 1. 모델 로드

Hugging Face에서 모델과 토크나이저를 불러옵니다.

In [5]:
def _get_layers(model):
    for attr in ["model.layers", "transformer.h"]:
        obj = model
        try:
            for a in attr.split("."): obj = getattr(obj, a)
            return obj
        except AttributeError:
            continue
    raise ValueError("블록을 찾을 수 없음")

DTYPE = dtype_from_str(EXP["runtime"]["dtype"])
TRC = bool(EXP["runtime"]["trust_remote_code"])

models, tokenizers = {}, {}

for tag, name in MODELS.items():
    print(f"Loading {tag}: {name} ...", flush=True)
    tokenizers[tag] = AutoTokenizer.from_pretrained(name, trust_remote_code=TRC)
    models[tag] = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=DTYPE,
        trust_remote_code=TRC,
    ).to(DEVICE).eval()
    print(f"  -> {tag} 로드 완료 ({DEVICE})", flush=True)

for tag, m in models.items():
    n_layers = len(_get_layers(m))
    d_model  = m.config.hidden_size
    print(f"  {tag}: layers={n_layers}, d_model={d_model}, params={sum(p.numel() for p in m.parameters())/1e6:.1f}M")

Loading non_reasoning: Qwen/Qwen2.5-0.5B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  -> reasoning 로드 완료 (cuda)
  non_reasoning: layers=24, d_model=896, params=494.0M
  reasoning: layers=28, d_model=1024, params=751.6M


## 2. 활성화 캡처 함수

순전파 시 레이어별 잔차 스트림을 모읍니다.

In [6]:
def capture_activations(model, tokenizer, texts, device=DEVICE, max_length=None):
    """모든 레이어의 hidden state를 캡처.

    Returns: dict[int, Tensor(total_tokens, d_model)]
    """
    max_length = max_length if max_length is not None else int(EXP["runtime"]["max_length"])
    layers = _get_layers(model)
    n_layers = len(layers)
    storage = {}

    def hook_fn(name):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            storage[name] = h.detach().cpu()
        return hook

    handles = [layers[i].register_forward_hook(hook_fn(i)) for i in range(n_layers)]

    collected = {i: [] for i in range(n_layers)}
    with torch.no_grad():
        for text in tqdm(texts, desc=f"forward {n_layers}L", leave=True):
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            enc = {k: v.to(device) for k, v in enc.items()}
            model(**enc)
            for i in range(n_layers):
                collected[i].append(storage[i][0])  # (seq, d_model)

    for h in handles:
        h.remove()

    return {i: torch.cat(vecs, dim=0) for i, vecs in collected.items()}

print("capture_activations 준비됨")

capture_activations 준비됨


## 3. 전체 조건 실행

모델×프롬프트 조합마다 활성화를 수집합니다.

In [7]:
results = {}

for model_tag in MODELS:
    for prompt_tag, texts in PROMPTS.items():
        key = (model_tag, prompt_tag)
        print(f"▶ {model_tag} × {prompt_tag} ({len(texts)} prompts) ...", flush=True)
        results[key] = capture_activations(
            models[model_tag], tokenizers[model_tag], texts
        )

sample_key = list(results.keys())[0]
n_layers = len(results[sample_key])
d_model  = results[sample_key][0].shape[-1]
TARGET_LAYER = resolve_target_layer(EXP["analysis"]["target_layer"], n_layers)
print(f"\n완료: {len(results)} 조건, {n_layers} layers, d_model={d_model}")
print(f"TARGET_LAYER = {TARGET_LAYER}   (설정: {EXP['analysis']['target_layer']!r})")

▶ non_reasoning × intuitive (1 prompts) ...


forward 24L:   0%|          | 0/1 [00:00<?, ?it/s]

▶ non_reasoning × logical (1 prompts) ...


forward 24L:   0%|          | 0/1 [00:00<?, ?it/s]

▶ reasoning × intuitive (1 prompts) ...


forward 28L:   0%|          | 0/1 [00:00<?, ?it/s]

▶ reasoning × logical (1 prompts) ...


forward 28L:   0%|          | 0/1 [00:00<?, ?it/s]


완료: 4 조건, 24 layers, d_model=896
TARGET_LAYER = 12   (설정: 'mid')


## 4. 레이어별 L2 norm

조건별 평균 L2를 레이어 인덱스에 대해 그립니다.

In [20]:
layer_idx = list(range(n_layers))

fig = go.Figure()
for (mtag, ptag), acts in results.items():
    l2 = [acts[i].float().norm(dim=-1).mean().item() for i in layer_idx]
    fig.add_trace(go.Scatter(
        x=layer_idx, y=l2,
        mode="lines+markers",
        name=f"{mtag} / {ptag}",
    ))

fig.update_layout(
    title="레이어별 평균 L2 Norm (잔차 스트림)",
    xaxis_title="Layer",
    yaxis_title="Mean L2 Norm",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## 5. 레이어별 L2 차이 (P1 − P0)

같은 모델에서 두 프롬프트 그룹의 L2 차이를 봅니다.

In [21]:
_cols = max(1, len(MODELS))
fig = make_subplots(
    rows=1,
    cols=_cols,
    subplot_titles=list(MODELS.keys()),
)

for col, mtag in enumerate(MODELS, start=1):
    l2_p0 = [results[(mtag, P0)][i].float().norm(dim=-1).mean().item() for i in layer_idx]
    l2_p1 = [results[(mtag, P1)][i].float().norm(dim=-1).mean().item() for i in layer_idx]
    diff  = [b - a for a, b in zip(l2_p0, l2_p1)]

    fig.add_trace(
        go.Bar(x=layer_idx, y=diff, name=f"{mtag}: {P1} − {P0}"),
        row=1,
        col=col,
    )

fig.update_layout(
    title=f"질문 그룹에 따른 L2 차이 ({P1} − {P0})",
    template="plotly_white",
    height=400,
)
fig.show()

## 6. 뉴런 히트맵

타깃 레이어에서 조건별 평균 활성화(분산 상위 뉴런)를 히트맵으로 봅니다.

In [22]:
TOPN = int(EXP["analysis"]["heatmap_top_neurons"])

# d_model이 다르면 벡터 길이가 달라 분리
_rows = []
for mtag in MODELS:
    for ptag in PROMPTS:
        h = results[(mtag, ptag)][TARGET_LAYER].float()
        vec = h.mean(dim=0).numpy()
        _rows.append((f"{mtag}/{ptag}", vec))

_by_dim = {}
for lab, vec in _rows:
    d = int(vec.shape[0])
    _by_dim.setdefault(d, {"labels": [], "vectors": []})
    _by_dim[d]["labels"].append(lab)
    _by_dim[d]["vectors"].append(vec)

if len(_by_dim) == 1:
    _d = next(iter(_by_dim))
    _data = _by_dim[_d]
    matrix = np.stack(_data["vectors"])
    cond_labels = _data["labels"]
    var_across = matrix.var(axis=0)
    top_ix = np.argsort(var_across)[::-1][:TOPN]
    fig = go.Figure(data=go.Heatmap(
        z=matrix[:, top_ix],
        x=[str(i) for i in top_ix],
        y=cond_labels,
        colorscale="RdBu_r",
        zmid=0,
    ))
    fig.update_layout(
        title=f"Layer {TARGET_LAYER} — 뉴런별 평균 활성화 (d_model={_d}, 분산 상위 {TOPN})",
        xaxis_title="Neuron index",
        template="plotly_white",
        height=350,
    )
    fig.show()
else:
    _dims = sorted(_by_dim.keys())
    fig = make_subplots(
        rows=len(_dims),
        cols=1,
        subplot_titles=[f"d_model={d} (n={len(_by_dim[d]['labels'])})" for d in _dims],
        vertical_spacing=0.12,
    )
    for i, d in enumerate(_dims, start=1):
        _data = _by_dim[d]
        matrix = np.stack(_data["vectors"])
        cond_labels = _data["labels"]
        var_across = matrix.var(axis=0)
        top_ix = np.argsort(var_across)[::-1][:TOPN]
        fig.add_trace(
            go.Heatmap(
                z=matrix[:, top_ix],
                x=[str(j) for j in top_ix],
                y=cond_labels,
                colorscale="RdBu_r",
                zmid=0,
            ),
            row=i,
            col=1,
        )
    fig.update_layout(
        title_text=f"Layer {TARGET_LAYER} — d_model 별 서브플롯 (히트맵)",
        template="plotly_white",
        height=min(900, 120 + 180 * len(_dims)),
    )
    fig.show()


## 7. 차등 뉴런

primary 모델에서 P0 vs P1 평균 차이가 큰 뉴런을 봅니다.

In [23]:
TAG_PRI = EXP["tags_for_sae_plot"]["primary"]
TAG_BL = EXP["tags_for_sae_plot"]["baseline"]
KDIFF = int(EXP["analysis"]["differential_top_k"])

if TAG_PRI not in MODELS:
    raise KeyError(f"tags_for_sae_plot.primary={TAG_PRI!r} 가 MODELS 에 없습니다.")

h_pri_p0 = results[(TAG_PRI, P0)][TARGET_LAYER]
h_pri_p1 = results[(TAG_PRI, P1)][TARGET_LAYER]

diff_idx, diff_vals = differential_neurons(h_pri_p0, h_pri_p1, k=KDIFF)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[str(i) for i in diff_idx],
    y=[diff_vals[i] for i in diff_idx],
    marker_color="crimson",
))
fig.update_layout(
    title=f"[{TAG_PRI}] Layer {TARGET_LAYER} — |mean({P0}) − mean({P1})| 상위 {KDIFF} 뉴런",
    xaxis_title="Neuron index",
    yaxis_title=f"|mean({P0}) − mean({P1})|",
    template="plotly_white",
)
fig.show()

# baseline
if TAG_BL in MODELS and TAG_BL != TAG_PRI:
    h_bl_p0 = results[(TAG_BL, P0)][TARGET_LAYER]
    h_bl_p1 = results[(TAG_BL, P1)][TARGET_LAYER]

    fig2 = go.Figure()
    for label, ha, hb in [
        (TAG_PRI, h_pri_p0, h_pri_p1),
        (TAG_BL, h_bl_p0, h_bl_p1),
    ]:
        mean_a = ha.float().mean(dim=0).numpy()
        mean_b = hb.float().mean(dim=0).numpy()
        delta  = (mean_b - mean_a)[diff_idx]
        fig2.add_trace(go.Bar(
            x=[str(i) for i in diff_idx],
            y=delta.tolist(),
            name=label,
        ))

    fig2.update_layout(
        title=f"상위 {KDIFF} 차등 뉴런: ({P1} − {P0}) mean activation",
        xaxis_title="Neuron index",
        yaxis_title="Δ mean activation",
        barmode="group",
        template="plotly_white",
    )
    fig2.show()

## 8. 코사인 유사도

조건별 평균 벡터 사이 코사인 행렬입니다 (`d_model`이 다르면 차원별로 나뉩니다).

In [24]:
hiddens = {f"{m}/{p}": results[(m, p)][TARGET_LAYER] for m in MODELS for p in PROMPTS}
cos_by_dim = cosine_similarity_matrix(hiddens)

def _cosine_heatmap(labels, sim, title: str):
    return go.Figure(data=go.Heatmap(
        z=sim, x=labels, y=labels,
        colorscale="Blues", zmin=0, zmax=1,
        text=np.round(sim, 3).astype(str), texttemplate="%{text}",
    )).update_layout(
        title=title,
        template="plotly_white",
        height=450,
        width=550,
    )

if len(cos_by_dim) == 1:
    _d, (labels, sim) = next(iter(cos_by_dim.items()))
    fig = _cosine_heatmap(
        labels, sim,
        f"Layer {TARGET_LAYER} 조건 간 코사인 유사도 (d_model={_d})",
    )
    fig.show()
else:
    _dims = sorted(cos_by_dim.keys())
    fig = make_subplots(
        rows=len(_dims),
        cols=1,
        subplot_titles=[f"d_model={d}" for d in _dims],
        vertical_spacing=0.12,
    )
    for i, d in enumerate(_dims, start=1):
        labels, sim = cos_by_dim[d]
        fig.add_trace(
            go.Heatmap(
                z=sim, x=labels, y=labels,
                colorscale="Blues", zmin=0, zmax=1,
                text=np.round(sim, 3).astype(str), texttemplate="%{text}",
            ),
            row=i,
            col=1,
        )
    fig.update_layout(
        title_text=f"Layer {TARGET_LAYER} 조건 간 코사인 유사도 (d_model 별)",
        template="plotly_white",
        height=min(900, 120 + 380 * len(_dims)),
        width=550,
    )
    fig.show()


## 9. SAE 학습 (옵션)

수집한 활성으로 sparse autoencoder를 학습합니다.

In [13]:
# SAE_LAYER = TARGET_LAYER
# _st = EXP["analysis"]["sae_train"]

# train_acts = torch.cat([
#     results[(m, p)][SAE_LAYER] for m in MODELS for p in PROMPTS
# ], dim=0).float()

# print(f"SAE 학습 데이터: {train_acts.shape}  (tokens × d_model)")

# sae_cfg = SAETrainConfig(
#     d_input=train_acts.shape[-1],
#     d_hidden=train_acts.shape[-1] * 4,
#     k=int(_st["k"]),
#     lr=float(_st["lr"]),
#     l1_coeff=float(_st["l1_coeff"]),
#     batch_size=min(128, train_acts.shape[0]),
#     epochs=int(_st["epochs"]),
#     device=DEVICE,
# )

# trainer = SAETrainer(sae_cfg)
# history = trainer.train(train_acts)
# print(f"최종 loss: {history[-1]['loss']:.4f},  l0: {history[-1]['l0']:.0f}")

In [14]:
# # SAE 학습 곡선
# df_hist = pd.DataFrame(history)
# fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "L0 (활성 feature 수)"])
# fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["loss"], mode="lines", name="total"), row=1, col=1)
# fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["recon"], mode="lines", name="recon"), row=1, col=1)
# fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["l0"], mode="lines", name="l0"), row=1, col=2)
# fig.update_layout(template="plotly_white", height=350, title="SAE 학습 곡선")
# fig.show()

## 10. SAE feature 비교 (옵션)

학습한 SAE로 조건별 feature 평균을 비교합니다.

In [15]:
# sae = trainer.sae.eval()

# feature_means = {}
# for (mtag, ptag), acts in results.items():
#     h = acts[SAE_LAYER].float().to(DEVICE)
#     with torch.no_grad():
#         z = sae.encode(h)
#     feature_means[f"{mtag}/{ptag}"] = z.mean(dim=0).cpu().numpy()

# NSF = int(EXP["analysis"]["sae_top_features"])
# fm = np.stack(list(feature_means.values()))
# var = fm.var(axis=0)
# top_feat = np.argsort(var)[::-1][:NSF]

# fig = go.Figure(data=go.Heatmap(
#     z=fm[:, top_feat],
#     x=[str(i) for i in top_feat],
#     y=list(feature_means.keys()),
#     colorscale="Viridis",
# ))
# fig.update_layout(
#     title=f"SAE Layer {SAE_LAYER} — 조건 간 분산 상위 {NSF} Feature",
#     xaxis_title="SAE Feature index",
#     template="plotly_white",
#     height=350,
# )
# fig.show()

In [16]:
# z_pri_p1 = feature_means[f"{TAG_PRI}/{P1}"]
# z_pri_p0 = feature_means[f"{TAG_PRI}/{P0}"]
# delta_pri = z_pri_p1 - z_pri_p0

# if TAG_BL in MODELS and TAG_BL != TAG_PRI:
#     z_bl_p1 = feature_means[f"{TAG_BL}/{P1}"]
#     z_bl_p0 = feature_means[f"{TAG_BL}/{P0}"]
#     delta_bl = z_bl_p1 - z_bl_p0
#     specificity = np.abs(delta_pri) - np.abs(delta_bl)
#     top_specific = np.argsort(specificity)[::-1][:20]

#     fig = go.Figure()
#     fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_pri[top_specific], name=f"{TAG_PRI} Δ"))
#     fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_bl[top_specific], name=f"{TAG_BL} Δ"))
#     fig.update_layout(
#         title=f"[{TAG_PRI} vs {TAG_BL}] {P1}−{P0} 변화가 primary 에서 더 큰 SAE feature 20개",
#         xaxis_title="SAE Feature",
#         yaxis_title=f"mean z({P1}) − z({P0})",
#         barmode="group",
#         template="plotly_white",
#     )
#     fig.show()
# else:
#     top_specific = np.argsort(np.abs(delta_pri))[::-1][:20]
#     fig = go.Figure()
#     fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_pri[top_specific], name=f"{TAG_PRI}"))
#     fig.update_layout(
#         title=f"[{TAG_PRI}] 단일 모델 — {P1}−{P0} 변화 상위 20 SAE feature",
#         template="plotly_white",
#     )
#     fig.show()
